<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/exams/midterm/MID1_fairness_pipeline.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Midterm Capstone: Ethics, Fairness and Specification in Practice

## Project 1: A fairness audit and mitigation pipeline

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## The project

A lending model is trained on synthetic data in which a proxy feature carries information about a protected group. The example audits the model with group metrics and compares mitigation before, during and after training: reweighing, a parity-penalised logistic regression and group-specific thresholds [1, 2].

**Required extensions.** Add at least one further mitigation method, such as reductions-based in-processing [3], and evaluate all methods with confidence intervals over repeated splits. Extend the intersectional analysis, trace the complete fairness-accuracy frontier for two fairness definitions and discuss the results in the light of the impossibility theorems [4, 5]. Address the legal and ethical questions raised by using the protected attribute at decision time.

## What this example implements

The notebook generates the data, audits a group-blind baseline, applies three mitigation methods, traces a fairness-accuracy frontier and reports selection and true positive rates for intersectional subgroups.

Run the cells in order. The example is a baseline: the capstone extends it, evaluates the extensions and reports the results.

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Data with a proxy feature

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize

rng = np.random.default_rng(SEED)
n = 12000
A = rng.binomial(1, 0.35, n)                 # protected group, not used as a feature
B = rng.binomial(1, 0.5, n)                  # second attribute for intersectional analysis
income = rng.normal(50 - 8 * A, 12, n)
debt = rng.normal(20 + 4 * A, 8, n)
history = rng.normal(0, 1, n) - 0.3 * A
zip_risk = rng.normal(0.8 * A, 1, n)         # proxy feature correlated with the protected group
latent = 0.06 * income - 0.08 * debt + 0.9 * history + 0.2 * B + rng.normal(0, 1, n)
y = (latent > np.quantile(latent, 0.45)).astype(int)
X = np.c_[income, debt, history, zip_risk, B]
i_tr, i_te = train_test_split(np.arange(n), test_size=0.4, random_state=SEED, stratify=y)
scaler = StandardScaler().fit(X[i_tr])
Xtr, Xte = scaler.transform(X[i_tr]), scaler.transform(X[i_te])
ytr, yte, Atr, Ate, Bte = y[i_tr], y[i_te], A[i_tr], A[i_te], B[i_te]
print("repayment base rate by group:", round(y[A == 0].mean(), 3), round(y[A == 1].mean(), 3))

## 2. Audit and two mitigations

In [ ]:
def group_rates(yhat, ytrue, g):
    out = {}
    for v in (0, 1):
        m = g == v
        out[v] = {"selection": yhat[m].mean(), "TPR": yhat[m][ytrue[m] == 1].mean(), "FPR": yhat[m][ytrue[m] == 0].mean(),
                  "PPV": ytrue[m][yhat[m] == 1].mean() if (yhat[m] == 1).any() else np.nan}
    return out

def audit(yhat, ytrue=None, g=None):
    ytrue = yte if ytrue is None else ytrue
    g = Ate if g is None else g
    r = group_rates(yhat, ytrue, g)
    return {"accuracy": float(np.mean(yhat == ytrue)), "DP difference": r[1]["selection"] - r[0]["selection"],
            "TPR difference": r[1]["TPR"] - r[0]["TPR"], "FPR difference": r[1]["FPR"] - r[0]["FPR"], "PPV difference": r[1]["PPV"] - r[0]["PPV"]}

base = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
p_base = base.predict_proba(Xte)[:, 1]
results = {"baseline, group-blind": audit((p_base >= 0.5).astype(int))}

def reweighing_weights(a, yv):
    w = np.empty(len(yv))
    for av in (0, 1):
        for c in (0, 1):
            m = (a == av) & (yv == c)
            w[m] = np.mean(a == av) * np.mean(yv == c) / max(np.mean(m), 1e-12)
    return w

rw = LogisticRegression(max_iter=2000).fit(Xtr, ytr, sample_weight=reweighing_weights(Atr, ytr))
results["pre-processing: reweighing"] = audit((rw.predict_proba(Xte)[:, 1] >= 0.5).astype(int))

p_tr = base.predict_proba(Xtr)[:, 1]
target_tpr = np.mean(p_tr[ytr == 1] >= 0.5)
thr = {v: np.quantile(p_tr[(Atr == v) & (ytr == 1)], 1 - target_tpr) for v in (0, 1)}
results["post-processing: group thresholds"] = audit((p_base >= np.where(Ate == 1, thr[1], thr[0])).astype(int))
pd.DataFrame(results).T.round(3)

## 3. A parity-penalised model and its frontier

In [ ]:
Xb_tr, Xb_te = np.c_[Xtr, np.ones(len(Xtr))], np.c_[Xte, np.ones(len(Xte))]

def fit_fair(lam):
    def objective(w):
        s = 1 / (1 + np.exp(-Xb_tr @ w))
        ll = -np.mean(ytr * np.log(s + 1e-9) + (1 - ytr) * np.log(1 - s + 1e-9))
        d = s[Atr == 1].mean() - s[Atr == 0].mean()
        ds = s * (1 - s)
        g_d = Xb_tr[Atr == 1].T @ ds[Atr == 1] / (Atr == 1).sum() - Xb_tr[Atr == 0].T @ ds[Atr == 0] / (Atr == 0).sum()
        grad = Xb_tr.T @ (s - ytr) / len(ytr) + 2 * lam * d * g_d + 2e-4 * w
        return ll + lam * d ** 2 + 1e-4 * w @ w, grad
    return minimize(objective, np.zeros(Xb_tr.shape[1]), jac=True, method="L-BFGS-B").x

frontier = []
for lam in [0, 1, 3, 10, 30, 100]:
    yhat = (1 / (1 + np.exp(-Xb_te @ fit_fair(lam))) >= 0.5).astype(int)
    a = audit(yhat)
    frontier.append((lam, a["accuracy"], abs(a["DP difference"])))
    if lam == 30:
        results["in-processing: parity penalty"] = a
fr = pd.DataFrame(frontier, columns=["lambda", "accuracy", "abs DP difference"])
plt.plot(fr["abs DP difference"], fr["accuracy"], "o-")
for _, r in fr.iterrows():
    plt.annotate(f"{r['lambda']:g}", (r["abs DP difference"], r["accuracy"]), fontsize=8)
plt.xlabel("absolute demographic parity difference"); plt.ylabel("accuracy"); plt.title("Fairness-accuracy frontier of the penalised model"); plt.show()
pd.DataFrame(results).T.round(3)

## 4. Intersectional subgroups

In [ ]:
yhat = (p_base >= 0.5).astype(int)
rows = []
for av in (0, 1):
    for bv in (0, 1):
        m = (Ate == av) & (Bte == bv)
        rows.append({"A": av, "B": bv, "n": int(m.sum()), "selection": yhat[m].mean(), "TPR": yhat[m][yte[m] == 1].mean()})
pd.DataFrame(rows).round(3)

## Report and submission

The technical report states the question, explains the design choices, presents the experiments with the figures produced by the notebook and discusses the ethical and safety implications and the limitations of the results. It cites at least eight sources in square brackets, at least four of them from the course readings, and links to the final Colab notebook.

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the midterm capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 7, with the report and all code files attached or linked.

## References

[1] Kamiran, F., & Calders, T. (2012). Data preprocessing techniques for classification without discrimination. *Knowledge and Information Systems*, 33(1), 1-33. <https://doi.org/10.1007/s10115-011-0463-8>

[2] Hardt, M., Price, E., & Srebro, N. (2016). Equality of opportunity in supervised learning. In *Advances in Neural Information Processing Systems 29*.

[3] Agarwal, A., Beygelzimer, A., Dudík, M., Langford, J., & Wallach, H. (2018). A reductions approach to fair classification. In *Proceedings of the 35th International Conference on Machine Learning, PMLR 80* (pp. 60-69).

[4] Kleinberg, J., Mullainathan, S., & Raghavan, M. (2017). Inherent trade-offs in the fair determination of risk scores. In *8th Innovations in Theoretical Computer Science Conference (ITCS 2017), LIPIcs 67* (pp. 43:1-43:23). <https://doi.org/10.4230/LIPIcs.ITCS.2017.43>

[5] Chouldechova, A. (2017). Fair prediction with disparate impact: A study of bias in recidivism prediction instruments. *Big Data*, 5(2), 153-163. <https://doi.org/10.1089/big.2016.0047>